# 02 - Flat Flower Baseline

This vignette executes safe control-plane code for the one-layer Flower baseline.
The flat baseline is a comparison point: all six healthcare-network sites connect to
one `flat` SuperLink, instead of using regional hubs plus gateway SuperNodes.

In [1]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
ROOT

PosixPath('/Users/david/Documents/GitHub/project')

## 1. Regenerate Compose and inspect flat services

In [2]:
import subprocess
import sys
from pathlib import Path
import pandas as pd
import yaml

flat_compose = Path("reports/docker-compose-flat-preview.yml")
flat_compose.parent.mkdir(parents=True, exist_ok=True)
proc = subprocess.run(
    [sys.executable, "scripts/generate_compose.py", "--output", str(flat_compose)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(proc.stdout.strip())

compose = yaml.safe_load(flat_compose.read_text())
flat_services = sorted(name for name in compose["services"] if name.startswith("flat-"))
pd.DataFrame({"flat_service": flat_services})

Wrote reports/docker-compose-flat-preview.yml


,flat_service
0,flat-hospital-eu-01-superexec-clientapp
1,flat-hospital-eu-01-supernode
2,flat-hospital-eu-02-superexec-clientapp
3,flat-hospital-eu-02-supernode
4,flat-hospital-eu-03-superexec-clientapp
5,flat-hospital-eu-03-supernode
6,flat-hospital-na-01-superexec-clientapp
7,flat-hospital-na-01-supernode
8,flat-hospital-na-02-superexec-clientapp
9,flat-hospital-na-02-supernode


## 2. Execute the flat FL dry-run command

In [3]:
cmd = [
    sys.executable,
    "scripts/flat_fl_baseline.py",
    "--rounds",
    "1",
    "--batch-size",
    "8192",
    "--dry-run",
]
proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, check=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)

Running: flwr run . flat --stream --run-config 'level="regional" region="flat" 
global-round=1 init-checkpoint="/shared/checkpoints/flat/round_0.pt" 
output-checkpoint="/shared/checkpoints/flat/round_1.pt" num-server-rounds=1 
local-epochs=1 batch-size=8192 learning-rate=0.001 input-dim=78 
region-num-examples=1981517 fraction-train=1.0 fraction-evaluate=1.0 
min-train-nodes=6 min-evaluate-nodes=6 min-available-nodes=6'
Flat FL checkpoint: shared/checkpoints/flat/round_1.pt



## 3. Optional flat metrics

In [4]:
flat_metrics = Path("reports/flat_metrics_summary.csv")
if flat_metrics.exists():
    pd.read_csv(flat_metrics).head(12)
else:
    print("Flat metrics are not present. Run `make flat` and then evaluate the flat checkpoint to populate them.")

Flat metrics are not present. Run `make flat` and then evaluate the flat checkpoint to populate them.
